In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Target hardware engine:", device)

Target hardware engine: cuda


In [ ]:
train=r"E:\engg\Data\brain_tumor_data\Training"
test=r"E:\engg\Data\brain_tumor_data\Testing"

transform_pipeline = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [3]:
train_dataset = datasets.ImageFolder(root=train, transform=transform_pipeline)
test_dataset = datasets.ImageFolder(root=test, transform=transform_pipeline)

class_labels = train_dataset.classes
num_classes = len(class_labels)
print("Detected Tumor Classes:", class_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training items: {len(train_dataset)} | Testing items: {len(test_dataset)}")

Detected Tumor Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Training items: 5552 | Testing items: 1600


In [4]:
class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes):
        super(BrainTumorCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.fc1 = nn.Linear(128 * 18 * 18, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x) 
        return x

model = BrainTumorCNN(num_classes=len(class_labels)).to(device)
print(model)

BrainTumorCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=41472, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=4, bias=True)
)


In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [6]:
epochs = 20

for epoch in range(epochs):
    model.train() 
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = (correct_train / total_train) * 100
    
    print(f"Epoch {epoch+1}/{epochs} -> Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

Epoch 1/20 -> Loss: 0.7138 | Accuracy: 71.81%
Epoch 2/20 -> Loss: 0.3384 | Accuracy: 87.55%
Epoch 3/20 -> Loss: 0.2071 | Accuracy: 92.71%
Epoch 4/20 -> Loss: 0.1320 | Accuracy: 95.03%
Epoch 5/20 -> Loss: 0.0800 | Accuracy: 97.26%
Epoch 6/20 -> Loss: 0.0581 | Accuracy: 98.16%
Epoch 7/20 -> Loss: 0.0231 | Accuracy: 99.23%
Epoch 8/20 -> Loss: 0.0268 | Accuracy: 99.14%
Epoch 9/20 -> Loss: 0.0208 | Accuracy: 99.24%
Epoch 10/20 -> Loss: 0.0363 | Accuracy: 98.94%
Epoch 11/20 -> Loss: 0.0320 | Accuracy: 98.87%
Epoch 12/20 -> Loss: 0.0269 | Accuracy: 99.05%
Epoch 13/20 -> Loss: 0.0199 | Accuracy: 99.51%
Epoch 14/20 -> Loss: 0.0118 | Accuracy: 99.50%
Epoch 15/20 -> Loss: 0.0185 | Accuracy: 99.69%
Epoch 16/20 -> Loss: 0.0141 | Accuracy: 99.69%
Epoch 17/20 -> Loss: 0.0009 | Accuracy: 99.98%
Epoch 18/20 -> Loss: 0.0001 | Accuracy: 100.00%
Epoch 19/20 -> Loss: 0.0001 | Accuracy: 100.00%
Epoch 20/20 -> Loss: 0.0000 | Accuracy: 100.00%


In [10]:
model.eval() 
correct_test = 0
total_test = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()

final_accuracy = (correct_test / total_test) * 100
print(f"\nFinal Test Accuracy: {final_accuracy:.2f}%")


Final Test Accuracy: 90.69%


In [8]:
torch.save(model.state_dict(), "brain_tumor_cnn.pth")

In [9]:
print(train_dataset.classes)

['glioma', 'meningioma', 'notumor', 'pituitary']
